In [ ]:
import torch
import torch.nn as nn
import tiktoken
import torch.nn.functional as F
import requests
from torch.utils.data import DataLoader

In [ ]:
GPT_CONFIG_124M = {
    "vocab_size" : 50257,
    "n_layers" : 12,
    "n_heads" : 12,
    "emb_dim" : 768,
    "dropout" : 0.1,
    "qkv_bias" : False,
    "context_length" : 1024
}

In [ ]:
class GPTModel(nn.Module):
  def __init__(self, cfg):
    super().__init__()

    self.token_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
    self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
    self.dropout_layer = nn.Dropout(cfg["dropout"])

    self.trf = nn.Sequential(
        *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
    )

    self.final_layer = LayerNorm(cfg)
    self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

  def forward(self, idx):
    batch_size, seq_len = idx.shape
    token_embd = self.token_emb(idx)
    pos_indices = torch.arange(seq_len, device = idx.device)
    pos_embd = self.pos_emb(pos_indices)
    x = token_embd + pos_embd
    x = self.dropout_layer(x)
    x = self.trf(x)
    x = self.final_layer(x)
    logits = self.out_head(x)
    return logits


In [ ]:
class TransformerBlock(nn.Module):
  def __init__(self, cfg):
    super().__init__()

    self.norm1 = LayerNorm(cfg)
    self.att = MultiHeadAttention(cfg)
    self.dropout_shortcut = nn.Dropout(cfg["dropout"])

    self.norm2 = LayerNorm(cfg)
    self.ff = ForwardLayer(cfg)

  def forward(self, x):
    shortcut_input = x
    x = self.norm1(x)
    x = self.att(x)
    x = self.dropout_shortcut(x)
    x = x + shortcut_input

    shortcut_input = x
    x = self.norm2(x)
    x = self.ff(x)
    x = self.dropout_shortcut(x)
    x = x + shortcut_input

    return x

In [ ]:
class LayerNorm(nn.Module):
  def __init__(self, cfg):
    super().__init__()

    self.scale = nn.Parameter(torch.ones(cfg["emb_dim"]))
    self.shift = nn.Parameter(torch.zeros(cfg["emb_dim"]))
    self.epsilon = 1e-5

  def forward(self, x):
    mean = x.mean(dim=-1, keepdim=True)
    var = x.var(dim=-1, keepdim=True)
    x_norm = (x - mean) / torch.sqrt(var + self.epsilon)
    return x_norm * self.scale + self.shift

In [ ]:
class ForwardLayer(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    self.layers = nn.Sequential(
        nn.Linear(cfg["emb_dim"], 4*cfg["emb_dim"]),
        nn.GELU(),
        nn.Linear(4*cfg["emb_dim"], cfg["emb_dim"])
    )

  def forward(self, x):
    return self.layers(x)

In [ ]:
class MultiHeadAttention(nn.Module):
  def __init__(self, cfg):
    super().__init__()

    self.d_out = cfg["emb_dim"]
    self.head_dim = cfg["emb_dim"] // cfg["n_heads"]
    self.num_heads = cfg["n_heads"]

    self.Query = nn.Linear(cfg["emb_dim"], cfg["emb_dim"], bias = cfg["qkv_bias"])
    self.Key = nn.Linear(cfg["emb_dim"], cfg["emb_dim"], bias = cfg["qkv_bias"])
    self.Value = nn.Linear(cfg["emb_dim"], cfg["emb_dim"], bias = cfg["qkv_bias"])
    self.out_proj = nn.Linear(cfg["emb_dim"], cfg["emb_dim"], bias = cfg["qkv_bias"])

    self.dropout = nn.Dropout(cfg["dropout"])
    mask = torch.triu(torch.ones(cfg["context_length"], cfg["context_length"]), diagonal=1)
    self.register_buffer("mask", mask)

  def forward(self, x):
    b, token_len, d_in = x.shape

    Queries = self.Query(x)#B,T,d_in
    Keys = self.Key(x)
    Values = self.Value(x)

    #B,T,d_out -> B,T,H, d_out
    Queries = Queries.view(b, token_len, self.num_heads, self.head_dim)
    Keys = Keys.view(b, token_len, self.num_heads, self.head_dim)
    Values = Values.view(b, token_len, self.num_heads, self.head_dim)

    #B,T,H, d_out -> B,H,T, d_out
    Queries = Queries.transpose(1,2)
    Keys = Keys.transpose(1,2)
    Values = Values.transpose(1,2)

    #B,H,T, d_out @ B,H,d_out, T -> B,H,T, T
    attn_scores = Queries @ Keys.transpose(-1, -2)
    d_k = Keys.shape[-1] ** 0.5

    causal_mask = self.mask[:token_len, :token_len].bool()
    attn_scores = attn_scores.masked_fill(causal_mask, -torch.inf)

    attn_weights = torch.softmax(attn_scores/d_k, dim=-1)
    attn_weights = self.dropout(attn_weights)

    #B,H,T, T @ B,H,T, d_out -> B,H,T, d_out
    context_vec = attn_weights @ Values

    #B,H,T, d_out -> B,T,H,d_out
    context_vec = context_vec.transpose(1,2)

    #B,T,H,d_out -> B,T*H,d_out
    context_vec = context_vec.contiguous().view(b, token_len, self.d_out)

    context_vec = self.out_proj(context_vec)
    return context_vec

In [ ]:
"""def generate_text_simple(model, idx, max_new_tokens, context_size):
  for _ in range(max_new_tokens):
    idx_cond = idx[:, -context_size:]

    with torch.no_grad():
      logits = model(idx_cond)

      logits = logits[:, -1, :]
      idx_next = torch.argmax(logits, dim=-1, keepdim=True)
      idx = torch.cat((idx, idx_next), dim=-1)
  return idx"""

'def generate_text_simple(model, idx, max_new_tokens, context_size):\n  for _ in range(max_new_tokens):\n    idx_cond = idx[:, -context_size:]\n\n    with torch.no_grad():\n      logits = model(idx_cond)\n\n      logits = logits[:, -1, :]\n      idx_next = torch.argmax(logits, dim=-1, keepdim=True)\n      idx = torch.cat((idx, idx_next), dim=-1)\n  return idx'

In [ ]:
"""if __name__ == "__main__":
  torch.manual_seed(123)

  model = GPTModel(GPT_CONFIG_124M)
  model.eval()

  context_start = "Hello! there, How are you doing?"
  tokenizer = tiktoken.get_encoding("gpt2")
  print("Generating Text...")
  output_indices = generate_text_simple(model=model, idx=text_to_input(context_start, tokenizer), max_new_tokens = 10, context_size = GPT_CONFIG_124M["context_length"])
  print(f"Input Shape: {start_context.shape}")
  print(f"Output Shape: {output_indices.shape}")
  print(f"Generated Tokens: {input_to_text(output_indices, tokenizer)}")"""

'if __name__ == "__main__":\n  torch.manual_seed(123)\n\n  model = GPTModel(GPT_CONFIG_124M)\n  model.eval()\n\n  context_start = "Hello! there, How are you doing?"\n  tokenizer = tiktoken.get_encoding("gpt2")\n  print("Generating Text...")\n  output_indices = generate_text_simple(model=model, idx=text_to_input(context_start, tokenizer), max_new_tokens = 10, context_size = GPT_CONFIG_124M["context_length"])\n  print(f"Input Shape: {start_context.shape}")\n  print(f"Output Shape: {output_indices.shape}")\n  print(f"Generated Tokens: {input_to_text(output_indices, tokenizer)}")'

In [ ]:
def text_to_input(text, tokenizer):
  encoded_text = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
  encoded_text = torch.tensor(encoded_text)
  encoded_text = encoded_text.unsqueeze(dim=0)
  return encoded_text

In [ ]:
def input_to_text(idx, tokenizer):
  idx = idx.squeeze(dim=0)
  idx_list = idx.tolist()
  return tokenizer.decode(idx_list)

In [ ]:
url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/refs/heads/main/ch02/01_main-chapter-code/the-verdict.txt"
temp = requests.get(url)
temp.raise_for_status()
raw_text = temp.text

In [ ]:
class GPT_Dataset():
  def __init__(self, text, tokenizer, context_length, stride):
    super().__init__()
    self.input_ids = []
    self.target_ids = []
    token_ids = tokenizer.encode(text)

    for idx in range(0, len(token_ids)-context_length, stride):
      temp_inputs = token_ids[idx:idx+context_length]
      temp_targets = token_ids[idx+1:idx+context_length+1]
      self.input_ids.append(temp_inputs)
      self.target_ids.append(temp_targets)

  def __len__(self):
    return len(self.input_ids)

  def __getitem__(self, idx):
    return self.input_ids[idx], self.target_ids[idx]

In [ ]:
train_text = raw_text[:int(len(raw_text)*0.8)]
val_text = raw_text[int(len(raw_text)*0.8):]
tokenizer = tiktoken.get_encoding("gpt2")
train_dataset = GPT_Dataset(train_text, tokenizer, context_length=256, stride=256)
val_dataset = GPT_Dataset(val_text, tokenizer, context_length=256, stride=256)
train_dataloader = DataLoader(
    train_dataset,
    batch_size = 2,
    shuffle = True,
    drop_last = True,
    num_workers = 0
)
val_dataloader = DataLoader(
    val_dataset,
    batch_size = 2,
    shuffle = False,
    drop_last = False,
    num_workers = 0
)

In [ ]:
def cal_batch_loss(inputs, targets, model, device):
  inputs, targets = inputs.to(device), targets.to(device)
  model.to(device)
  logits = model(inputs)
  loss = F.cross_entropy(logits.flatten(0,1), targets.flatten(0))
  return loss


def cal_loss(dataloader, model, device, num_batches=None):
    total_loss = 0.0

    if len(dataloader) == 0:
        return float("nan")

    if num_batches is None:
        num_batches = len(dataloader)
    else:
        num_batches = min(num_batches, len(dataloader))

    for idx, (inputs, targets) in enumerate(dataloader):
        if idx < num_batches:
            batch_loss = cal_batch_loss(inputs, targets, model, device)
            total_loss += batch_loss.item()
        else:
            break

    return total_loss / num_batches

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
model = GPTModel(GPT_CONFIG_124M)

KeyError: ('context_length', 50257)